## Presentasi Bab VI v3 — kelompok penanda

**SPEC / DESIGN:** presentasi seluruh tabel dukungan v2 pada lebar 14 cm, dengan inti Bab IV dan tambahan dipisahkan. Jalankan hanya tiga sel berikut untuk menyiapkan PNG, PDF vektor, tabel satuan asli, dan pratinjau A4. Persona−N dan I−E diutamakan; I1–I3 tetap terpisah. Hasil lama dipertahankan. Status pengukuran masih kandidat.

**IMPLEMENTATION / CHECKLIST / TESTING / INTEGRATION:** baca manifest dan laporan docs/gender_lm_bab6_ringkas_20261007.md. Statistik v2 dibekukan; file baru ditulis ke run dan subfolder baru. Naskah tesis belum disinkronkan.


In [ ]:
CP_BENCHMARK = "ETHICS"
# Standalone presentation from the frozen support-v2 table; no inference is rerun.
from pathlib import Path
from datetime import datetime
import ast, hashlib, json, shutil, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import MaxNLocator
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
import nbformat

CP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "AGENTS.md").exists())
CP_ACTIVE = CP_ROOT / "notebooks/gender/linguistic marker" / CP_BENCHMARK
CP_NOTEBOOK = CP_ACTIVE / ("linguistic_marker_gender_gsm8k_direct_gender_v3.ipynb" if CP_BENCHMARK == "GSM8K" else "ETHICS_Gender_Linguistic_Markers_Analysis_v1_5_candidate.ipynb")
CP_SOURCE_ID = "20261004_232525_357456" if CP_BENCHMARK == "GSM8K" else "20261004_232242_586663"
CP_SOURCE = CP_ROOT / "result/analyses/linguistic_markers/runs" / f"gender_{CP_BENCHMARK.lower()}_support_v2_{CP_SOURCE_ID}"
CP_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
CP_RUN = CP_ROOT / "result/analyses/linguistic_markers/runs" / f"gender_{CP_BENCHMARK.lower()}_bab6_v3_{CP_ID}"
CP_RUN.mkdir(parents=True, exist_ok=False)
CP_STAGE = CP_RUN / "figures"
CP_STAGE.mkdir()
CP_DEST = CP_ACTIVE / "figures" / f"bab6_v3_{CP_ID}"
CP_FAMILY_SIZE = 378 if CP_BENCHMARK == "GSM8K" else 3960
def cp_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CP_INPUT_HASHES = {str(p): cp_hash(p) for p in CP_SOURCE.iterdir() if p.is_file()}
CP_OLD_FIGURES = {str(p): cp_hash(p) for p in (CP_ACTIVE / "figures").glob("*") if p.is_file()}
CP_DATA = pd.read_csv(CP_SOURCE / "comparisons_support_v2_candidate.csv", float_precision="round_trip", low_memory=False)
CP_DATA.insert(0, "source_row_id", np.arange(len(CP_DATA)))
CP_REGISTRY = json.loads((CP_SOURCE / "registry.json").read_text(encoding="utf-8"))
CP_REG = {r["marker_id"]: r for r in CP_REGISTRY}
CP_NB = nbformat.read(CP_NOTEBOOK, as_version=4)
# Reuse only literal label/model constants, without executing any legacy cells.
CP_LABELS = {}
for cell in CP_NB.cells:
    if cell.cell_type != "code":
        continue
    for node in ast.parse(cell.source).body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "RV_LABELS" for t in node.targets):
            CP_LABELS = ast.literal_eval(node.value)
assert CP_LABELS
CP_LABELS.update({
    "organization_explicit_step": "Langkah eksplisit", "organization_bullet": "Daftar berpoin",
    "organization_section_header": "Subjudul", "stance_assumption": "Asumsi",
    "stance_self_correction": "Koreksi diri", "connective_causal": "Sebab-akibat",
    "connective_contrastive": "Kontras", "connective_conditional": "Kondisi",
    "self_reference_first_person": "Orang pertama", "self_reference_inclusive_we": "We inklusif",
    "engagement_reader_directive": "Ajakan pembaca", "stance_contextual_hedge": "Hedge kontekstual",
    "stance_distancing": "Pendapat pihak lain", "affect_empathy_validation": "Empati / validasi",
    "connective_additive": "Penambahan", "connective_concessive": "Konsesi",
    "elaboration_code_gloss": "Penjelas ulang", "ai_identity_statement": "Identitas AI",
    "math_latex_display": "LaTeX display", "math_inline_arithmetic": "Aritmetika teks",
    "math_boxed_answer": "Boxed answer",
})
CP_MODELS = ["meta-llama_llama-3.1-8b-instruct", "meta-llama_llama-3.3-70b-instruct",
             "mistralai_ministral-8b-2512", "deepseek_deepseek-r1-distill-qwen-32b",
             "openai_gpt-oss-20b", "qwen_qwen3-next-80b-a3b-thinking"]
CP_NAMES = dict(zip(CP_MODELS, ["Llama 3.1 8B", "Llama 3.3 70B", "Ministral 8B", "DeepSeek R1 32B", "GPT-OSS 20B", "Qwen3 Next 80B"]))
CP_COLORS = dict(zip(CP_MODELS, ["#2463EB", "#8038DC", "#00946A", "#D52C37", "#C26A00", "#007E99"]))
CP_CONDITIONS = ["explicit_1", "implicit_1", "implicit_2", "implicit_3"]
CP_SHORT = dict(zip(CP_CONDITIONS, ["E", "I1", "I2", "I3"]))
CP_ROLES = [
    ("man_vs_neutral", "Man: persona − N", CP_CONDITIONS),
    ("woman_vs_neutral", "Woman: persona − N", CP_CONDITIONS),
    ("man_vs_explicit", "Man: I − E", CP_CONDITIONS[1:]),
    ("woman_vs_explicit", "Woman: I − E", CP_CONDITIONS[1:]),
    ("direct", "Man − Woman", CP_CONDITIONS),
]
CP_CHARACTER = ["word_count", "avg_sentence_length", "ttr_300"]
CP_GROUPS = [
    ("01_susunan", "Susunan respons", ["organization_explicit_step", "organization_bullet", "organization_section_header"], "inti"),
    ("02_kepastian", "Kepastian dan keraguan", ["stance_hedge", "stance_booster", "stance_assumption", "stance_self_correction"], "inti"),
    ("03_konektif", "Kata penghubung logis", ["connective_causal", "connective_contrastive", "connective_conditional"], "inti"),
    ("04_rujukan", "Rujukan diri dan ajakan pembaca", ["self_reference_first_person", "self_reference_inclusive_we", "engagement_reader_directive"], "inti"),
]
if CP_BENCHMARK == "ETHICS":
    CP_GROUPS += [
        ("05_konteks", "Konteks, pendapat, dan empati", ["stance_contextual_hedge", "stance_distancing", "affect_empathy_validation"], "inti_ethics"),
        ("06_konektif_ethics", "Penambahan dan konsesi", ["connective_additive", "connective_concessive"], "inti_ethics"),
    ]
CP_GROUPS += [("07_tambahan", "Penjelasan tambahan dan identitas AI", ["elaboration_code_gloss", "ai_identity_statement"], "tambahan")]
if CP_BENCHMARK == "GSM8K":
    CP_GROUPS += [("08_matematika", "Bahasa matematis", ["math_latex_display", "math_inline_arithmetic", "math_boxed_answer"], "domain")]
CP_SCOPES = [s for s in ["gsm8k", "commonsense", "deontology", "justice", "virtue", "gabungan"] if s in set(CP_DATA.subset)]
CP_KEYS = ["model_id", "subset", "role", "condition", "marker_id"]
assert not CP_DATA.duplicated(CP_KEYS).any()
assert set(CP_DATA.model_id) == set(CP_MODELS)
assert set(CP_DATA.family_size) == {CP_FAMILY_SIZE}
assert (CP_DATA.n_changed <= CP_DATA.n_pairs).all()
assert ((CP_DATA.p_holm < .05) & CP_DATA.p_holm.notna()).equals(CP_DATA.significant_display)
CP_CORE = CP_CHARACTER + [m for _, _, ms, status in CP_GROUPS if status.startswith("inti") for m in ms]
assert len(CP_CORE) == (16 if CP_BENCHMARK == "GSM8K" else 21)
assert len(set(CP_CORE)) == len(CP_CORE)
CP_MAP = []
for marker in CP_DATA.marker_id.drop_duplicates():
    groups = [(key, title, status) for key, title, ms, status in CP_GROUPS if marker in ms]
    if marker in CP_CHARACTER:
        group, title, status = "00_karakteristik", "Panjang dan variasi teks", "inti"
    elif groups:
        assert len(groups) == 1
        group, title, status = groups[0]
    else:
        group, title, status = "lampiran_v2", "Penanda domain / tambahan lengkap v2", "domain_tambahan"
    r = CP_REG.get(marker, {})
    CP_MAP.append(dict(marker_id=marker, label=CP_LABELS[marker], group=group, group_label=title,
        chapter_iv_status=status, is_core=marker in CP_CORE,
        theoretical_source=r.get("theoretical_source", "Biber (1988); definisi operasional notebook"),
        operational_definition=r.get("operational_definition", marker),
        custom_adaptation=r.get("custom_adaptation", True),
        source_measurement=r.get("measurement", "text_characteristic"),
        chapter_iv_reference="Bab IV - Perancangan.tex:1270-1290" if marker in CP_CORE else "Bab IV - Perancangan.tex:1247-1260",
        output_location="gambar ringkas" if group != "lampiran_v2" else "lampiran lengkap v2 dan tabel satuan asli"))
CP_MAP = pd.DataFrame(CP_MAP)
assert len(CP_MAP) == CP_DATA.marker_id.nunique()
CP_MAP.to_csv(CP_RUN / "marker_chapter_iv_mapping.csv", index=False, encoding="utf-8-sig")
CP_TABLE = CP_DATA.merge(CP_MAP[["marker_id", "label", "group", "chapter_iv_status", "is_core"]], on="marker_id", validate="many_to_one")
CP_TABLE["support_k_over_n"] = CP_TABLE.n_changed.astype(int).astype(str) + "/" + CP_TABLE.n_pairs.astype(int).astype(str)
CP_TABLE["mean_difference_original_unit"] = CP_TABLE.effect
CP_TABLE["native_unit_label"] = CP_TABLE.measurement_unit.map({
    "words": "kata", "words_per_sentence": "kata per kalimat", "ratio": "rasio",
    "occurrences_per_100_words": "kemunculan per 100 kata", "percentage_points": "poin persentase",
}).fillna(CP_TABLE.measurement_unit)
CP_TABLE["display_priority"] = np.where(CP_TABLE.role.eq("direct") | CP_TABLE.subset.eq("gabungan") | ~CP_TABLE.is_core, "pendamping", "utama")
finite = CP_TABLE[["left_mean", "right_mean", "effect"]].notna().all(axis=1)
cp_scale = np.where(CP_TABLE.measurement_unit.eq("percentage_points"), 100, 1)
CP_TABLE["left_mean_unit"] = np.where(CP_TABLE.measurement_unit.eq("percentage_points"), "proporsi 0–1", CP_TABLE.native_unit_label)
CP_TABLE["right_mean_unit"] = CP_TABLE.left_mean_unit
assert np.allclose((CP_TABLE.left_mean - CP_TABLE.right_mean)[finite] * cp_scale[finite], CP_TABLE.loc[finite, "effect"], atol=1e-10)
CP_TABLE.to_csv(CP_RUN / "all_comparisons_native_units.csv", index=False, encoding="utf-8-sig")
# Keep the original v2 figures indexed as an appendix, with their existing hashes.
CP_APPENDIX = pd.read_csv(CP_SOURCE / "figure_index.csv")
CP_APPENDIX["source_path"] = CP_APPENDIX.figure.map(lambda s: str(CP_SOURCE / "figures" / s))
assert all(cp_hash(p) == h for p, h in zip(CP_APPENDIX.source_path, CP_APPENDIX.sha256))
CP_APPENDIX.to_csv(CP_RUN / "appendix_v2_index.csv", index=False, encoding="utf-8-sig")
CP_INDEX, CP_LAYOUT, CP_HEAT_ROWS, CP_CHAR_ROWS = [], [], [], []
CP_CM = 1 / 2.54
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 8.5,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "pdf.fonttype": 42, "ps.fonttype": 42,
    "savefig.dpi": 240, "hatch.linewidth": .35})
print(f"{CP_BENCHMARK}: {len(CP_DATA):,} perbandingan v2; {len(CP_CORE)} penanda inti; run {CP_ID}", flush=True)


In [ ]:
def cp_scope(scope):
    return "GSM8K" if scope == "gsm8k" else "ETHICS · " + ("Gabungan (pendamping)" if scope == "gabungan" else scope.capitalize())

def cp_caption(scope, role_title, title, kind):
    common = (f"{cp_scope(scope)}. {title}; {role_title}. Semua enam model dan variasi yang tersedia ditampilkan. "
        "I1–I3 adalah variasi isi warmup yang terpisah, bukan tingkat implisit. "
        "k/n adalah pasangan berubah/tersedia; TP berarti seluruh selisih teramati nol, "
        "ND berarti tidak ada pasangan. Hasil bersyarat pada penjelasan yang tersedia. "
        f"Bintang memakai p Holm < 0,05 dengan keluarga v2 tetap {CP_FAMILY_SIZE:,} hipotesis/model. "
        "Pemisahan gambar tidak menghitung ulang koreksi. Status kamus tetap kandidat, "
        "belum validasi penuh independen. Marker leksikal tidak mengukur kemampuan penalaran secara langsung. ")
    if kind == "heatmap":
        common += ("Warna dan angka menunjukkan matched rank-biserial pada selisih nonnol; "
            "positif berarti sisi kiri lebih tinggi. Warna pekat tidak berarti mayoritas respons berubah. "
            "Arsir menandai 1–19 pasangan berubah sebagai kehati-hatian, bukan ambang validitas. "
            "TP memiliki efek peringkat tidak terdefinisi dan tidak diganti nol. "
            "[d] adalah frekuensi per 100 kata; [b] adalah hadir/tidak. "
            "Selisih dalam satuan asli, rerata kedua sisi, dan CI tersedia pada tabel pendamping. ")
    elif kind == "characteristic":
        common += ("Titik adalah selisih rata-rata dalam satuan asli; garis adalah CI bootstrap 95% marginal sumber, "
            "bukan CI rank-biserial dan bukan interval simultan. CI disembunyikan untuk n<2; "
            "segitiga menandai n<20. CI tidak mencakup ketidakpastian akibat penjelasan tidak tersedia. ")
    else:
        common += "Cakupan memakai n pasangan penjelasan dibagi jumlah soal, bukan commit rate jawaban. "
    if scope == "gabungan":
        common += "Gabungan berbobot pasangan tersedia, bukan rata-rata efek subset atau kontribusi subset yang sama. "
    return common

CP_PROOF = PdfPages(CP_RUN / "pratinjau_A4_ukuran_asli.pdf")
CP_PROOF_COUNT = 0
CP_PROOF_FILES = []

def cp_finish(fig, stem, scope, role, kind, group, title, caption, nrows=0, proof=False):
    global CP_PROOF_COUNT
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    fw, fh = fig.bbox.width, fig.bbox.height
    outside = []
    for artist in fig.findobj(matplotlib.text.Text):
        if not artist.get_visible() or not artist.get_text():
            continue
        box = artist.get_window_extent(renderer)
        if box.x0 < -1 or box.y0 < -1 or box.x1 > fw + 1 or box.y1 > fh + 1:
            outside.append(artist.get_text())
    assert not outside, (stem, "text outside canvas", outside)
    width_cm, height_cm = fig.get_size_inches() * 2.54
    assert abs(width_cm - 14) < 1e-6 and height_cm <= 20.5
    png, pdf = CP_STAGE / f"{stem}.png", CP_STAGE / f"{stem}.pdf"
    assert not png.exists() and not pdf.exists()
    fig.savefig(png, facecolor="white", dpi=240)
    fig.savefig(pdf, facecolor="white", metadata={"Title": title, "Subject": "Gender linguistic markers; frozen support v2"})
    CP_INDEX.append(dict(figure=png.name, pdf=pdf.name, subset=scope, role=role, kind=kind,
        group=group, title=title, caption=caption, width_cm=width_cm, height_cm=height_cm,
        display_priority="pendamping" if role == "direct" or scope == "gabungan" or group in ("07_tambahan", "08_matematika") else "utama",
        source_table="all_comparisons_native_units.csv", png_sha256=cp_hash(png), pdf_sha256=cp_hash(pdf)))
    if proof:
        # Place the very same vector artists into the thesis text block, at 1:1 scale.
        axes_positions = [(ax, ax.get_position().frozen()) for ax in fig.axes]
        text_positions = [(tx, tx.get_position()) for tx in fig.texts]
        sx, sy = 14 / 21, height_cm / 29.7
        ox, oy = 4 / 21, (29.7 - 3 - height_cm) / 29.7
        fig.set_size_inches(21 * CP_CM, 29.7 * CP_CM)
        for ax, pos in axes_positions:
            ax.set_position([ox + pos.x0 * sx, oy + pos.y0 * sy, pos.width * sx, pos.height * sy])
        for tx, (x, y) in text_positions:
            tx.set_position((ox + x * sx, oy + y * sy))
        fig.add_artist(Rectangle((4/21, 3/29.7), 14/21, 23.7/29.7, transform=fig.transFigure,
            fill=False, edgecolor="#CBD5E1", linewidth=.5, linestyle="--"))
        proof_caption = ("Pratinjau ukuran asli · lebar gambar 14 cm. " + title + ".\n"
            "I1–I3: variasi warmup, bukan tingkatan. " +
            ("Warna pekat perlu dibaca bersama k/n." if kind == "heatmap" else "Selisih dihitung pada pasangan penjelasan tersedia."))
        fig.text(4/21, oy-.012, "\n".join(textwrap.fill(line, width=87) for line in proof_caption.splitlines()), ha="left", va="top", fontsize=8, linespacing=1.4)
        CP_PROOF.savefig(fig, facecolor="white")
        proof_name = f"pratinjau_A4_{CP_PROOF_COUNT + 1:02d}.png"
        fig.savefig(CP_RUN / proof_name, dpi=120, facecolor="white")
        CP_PROOF_FILES.append(dict(file=proof_name, source=png.name, width_cm=14))
        CP_PROOF_COUNT += 1
    plt.close(fig)
    if len(CP_INDEX) % 10 == 0:
        print(f"{CP_BENCHMARK}: {len(CP_INDEX)} gambar PNG + PDF selesai", flush=True)

def cp_heatmap(scope, role, role_title, conditions, group, group_title, markers, status):
    selected = CP_DATA[CP_DATA.subset.eq(scope) & CP_DATA.role.eq(role)].set_index(["model_id", "marker_id", "condition"])
    rows, cols = len(markers), len(conditions)
    height_cm = 6.0 + 2.7 * rows
    fig, axes = plt.subplots(3, 2, figsize=(14*CP_CM, height_cm*CP_CM), sharey=True)
    fig.subplots_adjust(left=.20, right=.995, top=.855, bottom=.17, wspace=.03, hspace=.48)
    cmap, norm = plt.get_cmap("RdBu_r"), Normalize(-1, 1)
    annotations = []
    for ax, model in zip(axes.flat, CP_MODELS):
        for y, marker in enumerate(markers):
            for x, condition in enumerate(conditions):
                r = selected.loc[(model, marker, condition)]
                k, n = int(r.n_changed), int(r.n_pairs)
                v = r.matched_rank_biserial
                if n == 0:
                    face, first, color = "#FFF0D6", "ND", "#172033"
                    assert np.isnan(v)
                elif k == 0:
                    face, first, color = "#E8EDF3", "TP", "#172033"
                    assert np.isnan(v)
                else:
                    assert np.isfinite(v)
                    face, first = cmap(norm(v)), f"{v:+.2f}" + ("*" if r.significant_display else "")
                    color = "white" if abs(v) > .6 else "#172033"
                ax.add_patch(Rectangle((x-.5,y-.5),1,1,facecolor=face,edgecolor="white",linewidth=.35))
                if 0 < k < 20:
                    ax.add_patch(Rectangle((x-.5,y-.5),1,1,facecolor="none",edgecolor="#999999",hatch="///",linewidth=0,alpha=.5))
                t1=ax.text(x,y-.20,first,ha="center",va="center",fontsize=8,color=color)
                t2=ax.text(x,y+.23,f"{k}/{n}",ha="center",va="center",fontsize=7,color=color)
                annotations.append((ax,x,y,t1,t2))
                CP_HEAT_ROWS.append(int(r.source_row_id))
        ax.set_xlim(-.5, cols-.5); ax.set_ylim(rows-.5,-.5)
        suffix = "−N" if role.endswith("neutral") else "−E" if role.endswith("explicit") else ""
        ax.set_xticks(range(cols), [CP_SHORT[c]+suffix for c in conditions])
        ax.set_yticks(range(rows), [textwrap.fill(CP_LABELS[m]+(" [b]" if CP_REG[m]["measurement"]=="binary" else " [d]"), width=16) for m in markers])
        ax.tick_params(axis="both",length=0,pad=3)
        ax.set_title(CP_NAMES[model],fontweight="bold",pad=6)
        for spine in ax.spines.values():
            spine.set_visible(False)
    core_label = "Inti" if status.startswith("inti") else "Tambahan"
    title = f"{cp_scope(scope)} · {role_title}"
    fig.suptitle(title+"\n"+group_title+" · "+core_label, fontsize=10,fontweight="bold",y=.985)
    cax=fig.add_axes([.3,.108,.6,.014])
    cb=fig.colorbar(ScalarMappable(norm=norm,cmap=cmap),cax=cax,orientation="horizontal",ticks=[-1,-.5,0,.5,1])
    cb.solids.set_rasterized(False)
    cb.ax.tick_params(labelsize=7,length=2,pad=1)
    fig.text(.03,.069,"r: rank-biserial · positif: sisi kiri lebih tinggi · k/n: berubah/tersedia",fontsize=7)
    fig.text(.03,.042,"* p Holm < 0,05 · arsir: k=1–19 · TP: tak berubah · ND: tak tersedia",fontsize=7)
    fig.text(.03,.015,"[d] per 100 kata · [b] hadir/tidak · I1–I3: variasi warmup · Kandidat",fontsize=7)
    fig.canvas.draw()
    renderer=fig.canvas.get_renderer()
    for ax,x,y,t1,t2 in annotations:
        cell_w=abs(ax.transData.transform((x+.5,y))[0]-ax.transData.transform((x-.5,y))[0])
        assert max(t1.get_window_extent(renderer).width,t2.get_window_extent(renderer).width) < cell_w*.99, (title,"cell text too wide",t2.get_text(),cell_w,t2.get_window_extent(renderer).width)
        CP_LAYOUT.append(dict(kind="cell",label=t2.get_text(),font_pt=7,cell_width_px=cell_w,
            text_width_px=t2.get_window_extent(renderer).width))
    proof = ((scope in ["gsm8k","commonsense"] and group=="02_kepastian" and role in ["man_vs_neutral","woman_vs_explicit"])
        or (scope=="justice" and group=="05_konteks" and role=="man_vs_neutral"))
    stem=f"{scope}__{CP_ROLES.index((role,role_title,conditions))+1:02d}_{role}__{group}"
    cp_finish(fig,stem,scope,role,"heatmap",group,title+" · "+group_title,
        cp_caption(scope,role_title,group_title,"heatmap"),rows,proof)

CP_CHAR_LABELS = {
    "word_count": "Selisih rata-rata jumlah kata (kata)",
    "avg_sentence_length": "Selisih rata-rata panjang kalimat (kata/kalimat)",
    "ttr_300": "Selisih rata-rata TTR-300 (rasio)",
}
CP_CHAR_SHORT = {"word_count":"Jumlah kata","avg_sentence_length":"Panjang kalimat","ttr_300":"TTR-300"}
def cp_characteristic(scope,role,role_title,conditions,marker):
    part=CP_DATA[CP_DATA.subset.eq(scope)&CP_DATA.role.eq(role)&CP_DATA.marker_id.eq(marker)]
    # Freeze an identical scale across all roles and conditions of this subset/outcome.
    all_part=CP_DATA[CP_DATA.subset.eq(scope)&CP_DATA.marker_id.eq(marker)]
    limits=all_part[["effect","ci_display_low","ci_display_high"]].to_numpy().ravel()
    limits=limits[np.isfinite(limits)]
    bound=max(abs(limits)) if len(limits) else 1
    bound=max(bound*1.12,.001)
    fig,axes=plt.subplots(2,2,figsize=(14*CP_CM,15*CP_CM),sharey=True)
    fig.subplots_adjust(left=.205,right=.89,top=.855,bottom=.21,wspace=.6,hspace=.40)
    for ax,condition in zip(axes.flat,conditions):
        values=part[part.condition.eq(condition)].set_index("model_id")
        for y,model in enumerate(CP_MODELS):
            r=values.loc[model]; n,k=int(r.n_pairs),int(r.n_changed)
            if np.isfinite(r.effect):
                if np.isfinite(r.ci_display_low):
                    ax.plot([r.ci_display_low,r.ci_display_high],[y,y],color=CP_COLORS[model],lw=1.1,ls="--" if n<20 else "-")
                ax.scatter(r.effect,y,color=CP_COLORS[model],s=18,marker="^" if n<20 else "o",zorder=3)
                if r.significant_display:
                    ax.annotate("*",(r.effect,y),xytext=(0,4),textcoords="offset points",ha="center",fontsize=8)
                state=" TP" if k==0 else ""
            else:
                ax.text(0,y,"ND",ha="center",va="center",fontsize=8);state=""
            ax.text(1.025,y,f"{k}/{n}"+state,transform=ax.get_yaxis_transform(),va="center",fontsize=7)
            CP_CHAR_ROWS.append(int(r.source_row_id))
        ax.text(1.025,1.02,"k/n",transform=ax.transAxes,fontsize=7)
        ax.set_xlim(-bound,bound);ax.set_ylim(5.6,-.6)
        ax.set_yticks(range(6),[CP_NAMES[m] for m in CP_MODELS],fontsize=7.2)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.tick_params(axis="x",labelsize=7)
        ax.set_title(CP_SHORT[condition]+("−N" if role.endswith("neutral") else "−E" if role.endswith("explicit") else ""),fontweight="bold")
        ax.axvline(0,color="#94A3B8",lw=.7);ax.grid(axis="x",color="#E2E8F0",lw=.5)
        ax.spines[["top","right"]].set_visible(False)
    if len(conditions)==3:
        axes.flat[-1].set_visible(False)
    title=f"{cp_scope(scope)} · {role_title}"
    fig.suptitle(title+"\n"+CP_CHAR_SHORT[marker],fontsize=10,fontweight="bold",y=.975)
    fig.text(.5,.145,CP_CHAR_LABELS[marker],ha="center",fontsize=8.5)
    fig.text(.03,.097,"Titik: selisih rata-rata · garis: CI 95% · k/n: berubah/tersedia",fontsize=7)
    fig.text(.03,.068,"* p Holm < 0,05 · segitiga: n<20 · CI ditampilkan jika n≥2",fontsize=7)
    fig.text(.03,.039,"TP: tak berubah · ND: tak tersedia · positif: sisi kiri lebih tinggi",fontsize=7)
    fig.text(.03,.012,"I1–I3: variasi warmup · Kandidat · skala sama antarperbandingan",fontsize=7)
    proof=scope in ["gsm8k","commonsense"] and role=="man_vs_neutral" and marker=="word_count"
    stem=f"{scope}__{CP_ROLES.index((role,role_title,conditions))+1:02d}_{role}__00_{marker}"
    cp_finish(fig,stem,scope,role,"characteristic","00_karakteristik",title+" · "+CP_CHAR_SHORT[marker],
        cp_caption(scope,role_title,CP_CHAR_LABELS[marker],"characteristic"),proof=proof)

def cp_coverage(scope,role,role_title,conditions):
    part=CP_DATA[CP_DATA.subset.eq(scope)&CP_DATA.role.eq(role)&CP_DATA.marker_id.eq("word_count")]
    fig,ax=plt.subplots(figsize=(14*CP_CM,7.5*CP_CM))
    fig.subplots_adjust(left=.25,right=.96,top=.76,bottom=.22)
    for y,model in enumerate(CP_MODELS):
        for x,c in enumerate(conditions):
            r=part[part.model_id.eq(model)&part.condition.eq(c)].iloc[0]
            n,total=int(r.n_pairs),int(r.total_submitted_questions)
            pct=100*n/total
            ax.add_patch(Rectangle((x-.5,y-.5),1,1,facecolor=plt.get_cmap("Blues")(pct/100),edgecolor="white"))
            ax.text(x,y,f"{pct:.1f}%\n{n}/{total}",ha="center",va="center",fontsize=8,
                color="white" if pct>60 else "#172033")
    ax.set_xlim(-.5,len(conditions)-.5);ax.set_ylim(5.5,-.5)
    ax.set_xticks(range(len(conditions)),[CP_SHORT[c] for c in conditions])
    ax.set_yticks(range(6),[CP_NAMES[m] for m in CP_MODELS],fontsize=8)
    ax.tick_params(length=0)
    title=f"{cp_scope(scope)} · {role_title}"
    fig.suptitle(title+"\nCakupan pasangan penjelasan",fontsize=10,fontweight="bold",y=.96)
    fig.text(.04,.10,"Isi sel: cakupan (%) dan pasangan tersedia / soal per skenario.",fontsize=7.5)
    fig.text(.04,.035,"Denominator cakupan berbeda dari k/n. I1–I3: variasi warmup · Kandidat",fontsize=7)
    stem=f"{scope}__{CP_ROLES.index((role,role_title,conditions))+1:02d}_{role}__00_cakupan"
    cp_finish(fig,stem,scope,role,"coverage","00_cakupan",title+" · Cakupan",
        cp_caption(scope,role_title,"Cakupan pasangan penjelasan","coverage"))

for cp_s in CP_SCOPES:
    for cp_role,cp_title,cp_conditions in CP_ROLES:
        cp_coverage(cp_s,cp_role,cp_title,cp_conditions)
        for cp_marker in CP_CHARACTER:
            cp_characteristic(cp_s,cp_role,cp_title,cp_conditions,cp_marker)
        for cp_group,cp_group_title,cp_markers,cp_status in CP_GROUPS:
            cp_heatmap(cp_s,cp_role,cp_title,cp_conditions,cp_group,cp_group_title,cp_markers,cp_status)
CP_PROOF.close()
CP_INDEX=pd.DataFrame(CP_INDEX)
CP_INDEX.to_csv(CP_RUN/"figure_index.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(CP_LAYOUT).to_csv(CP_RUN/"cell_layout_checks.csv",index=False,encoding="utf-8-sig")
print(f"Render selesai: {len(CP_INDEX)} PNG + PDF; {CP_PROOF_COUNT} halaman contoh A4.",flush=True)


In [ ]:
# Validate the presentation before publishing to a new local figures subfolder.
cp_expected = len(CP_SCOPES)*len(CP_ROLES)*(4+len(CP_GROUPS))
assert len(CP_INDEX)==cp_expected
assert len(set(CP_INDEX.figure))==len(CP_INDEX)
cp_heat_markers={m for _,_,ms,_ in CP_GROUPS for m in ms}
assert sorted(CP_HEAT_ROWS)==sorted(CP_DATA.loc[CP_DATA.marker_id.isin(cp_heat_markers),"source_row_id"].tolist())
assert sorted(CP_CHAR_ROWS)==sorted(CP_DATA.loc[CP_DATA.marker_id.isin(CP_CHARACTER),"source_row_id"].tolist())
for row in CP_INDEX.itertuples():
    with Image.open(CP_STAGE/row.figure) as im:
        im.load()
        assert im.width >= 1300 and im.height >= 700
    assert cp_hash(CP_STAGE/row.figure)==row.png_sha256
    assert cp_hash(CP_STAGE/row.pdf)==row.pdf_sha256
    pdf_bytes=(CP_STAGE/row.pdf).read_bytes()
    assert pdf_bytes.startswith(b"%PDF")
    assert b"/Subtype /Image" not in pdf_bytes, ("Raster found in vector PDF",row.pdf)
cp_roundtrip=pd.read_csv(CP_RUN/"all_comparisons_native_units.csv",float_precision="round_trip",low_memory=False)
pd.testing.assert_frame_equal(CP_DATA,cp_roundtrip[CP_DATA.columns],check_dtype=False)
assert all(cp_hash(p)==h for p,h in CP_INPUT_HASHES.items())
assert all(cp_hash(p)==h for p,h in CP_OLD_FIGURES.items())
assert CP_DEST.parent.resolve().is_relative_to(CP_ROOT.resolve())
assert not CP_DEST.exists()
shutil.copytree(CP_STAGE,CP_DEST)
assert all(cp_hash(CP_DEST/r.figure)==r.png_sha256 and cp_hash(CP_DEST/r.pdf)==r.pdf_sha256 for r in CP_INDEX.itertuples())
cp_manifest=dict(run=str(CP_RUN),benchmark=CP_BENCHMARK,source_run=str(CP_SOURCE),
    source_table="comparisons_support_v2_candidate.csv",active_figures=str(CP_DEST),
    analysis_version="support_v2_candidate",presentation_version="bab6_grouped_v3",
    scientific_status="candidate_pending_full_independent_validation",
    comparisons=len(CP_DATA),core_outcomes=len(CP_CORE),all_outcomes=CP_DATA.marker_id.nunique(),
    figure_count=len(CP_INDEX),vector_pdf_count=len(CP_INDEX),a4_proof_pages=CP_PROOF_COUNT,
    width_cm=14,max_height_cm=float(CP_INDEX.height_cm.max()),min_annotation_font_pt=7,
    support_notation="k/n: changed/available",holm_family_size=CP_FAMILY_SIZE,
    statistics_unchanged=True,source_hashes=CP_INPUT_HASHES,
    old_figure_hashes=CP_OLD_FIGURES,old_figures_preserved=True,
    appendix_v2_figures=len(CP_APPENDIX),filter_by_significance=False,
    thesis_files_modified=False,proof_files=CP_PROOF_FILES,
    implementation_notebook=str(CP_NOTEBOOK),implementation_sha256=cp_hash(CP_NOTEBOOK))
(CP_RUN/"manifest.json").write_text(json.dumps(cp_manifest,ensure_ascii=False,indent=2),encoding="utf-8")
(CP_RUN/"verification.json").write_text(json.dumps(dict(
    all_statistical_columns_equal=True,all_vector_pdfs_without_image_objects=True,
    core_markers_exactly_once=True,heatmap_rows=len(CP_HEAT_ROWS),characteristic_rows=len(CP_CHAR_ROWS),
    cell_text_bounds_checked=len(CP_LAYOUT),old_figures_unchanged=len(CP_OLD_FIGURES),
    pngs_decoded=len(CP_INDEX),physical_figure_width_cm=14,thesis_text_block_cm=[14,23.7],
    direct_comparison_role="pendamping",i_variants="separate_content_variants",
    print_readability="geometry checked; representative A4 pages require visual review"),
    ensure_ascii=False,indent=2),encoding="utf-8")
cp_cells=[c for c in CP_NB.cells if c.get("metadata",{}).get("presentation_section")=="gender_bab6_v3"]
cp_snapshot=nbformat.v4.new_notebook(cells=cp_cells,metadata=CP_NB.metadata)
nbformat.validate(cp_snapshot)
nbformat.write(cp_snapshot,CP_RUN/"presentation_notebook.ipynb")
cp_lines=[
    f"# {CP_BENCHMARK}: penanda linguistik gender — gambar ringkas Bab VI v3",
    "",f"Run: {CP_ID}. Status pengukuran **kandidat**; presentasi memakai seluruh statistik v2 yang dibekukan.",
    "",f"Folder gambar: {CP_DEST}.",f"Sumber: {CP_SOURCE}.",
    "",f"Tersedia {len(CP_INDEX)} PNG dan {len(CP_INDEX)} PDF vektor. Lebar setiap gambar 14 cm; "
       f"tinggi maksimum {CP_INDEX.height_cm.max():.1f} cm. Teks dukungan minimum 7 pt.",
    "", "## Cara meninjau", "",
    "1. Buka pratinjau_A4_ukuran_asli.pdf pada ukuran sebenarnya / cetak 100%, tanpa fit-to-page.",
    "2. Mulai dari persona−N dan I−E; I1–I3 adalah variasi warmup yang terpisah.",
    "3. figure_index.csv mencatat prioritas, kelompok, caption lengkap, dan pasangan PNG/PDF.",
    "4. all_comparisons_native_units.csv memuat rerata kedua sisi, selisih dalam satuan asli, CI, k/n, dan p Holm.",
    "5. appendix_v2_index.csv menunjuk versi lengkap yang tetap utuh. Gambar historis itu mempertahankan notasi lamanya.",
    "", "## Kelompok penanda dan batas interpretasi", "",
    f"Inti Bab IV: {len(CP_CORE)} ukuran termasuk tiga karakteristik teks. "
    "Penjelas ulang dan identitas AI dipisahkan sebagai tambahan; penanda domain memiliki kelompok tersendiri.",
    "",
    "Peta lengkap, definisi operasional, sumber teori, dan status adaptasi tersedia di marker_chapter_iv_mapping.csv. "
    "Peta ini menyelaraskan keanggotaan kelompok; tidak mengklaim kamus operasional sudah sepenuhnya tervalidasi. "
    "GSM8K v2 tidak memuat outcome pronomina gender yang disebut terpisah di Bab IV; tidak ditambahkan melalui pekerjaan visual ini.",
    "",
    "Heatmap inti maksimal empat baris dengan enam panel model. Positif berarti sisi kiri lebih tinggi. "
    "Warna pekat tidak menyatakan banyak respons berubah; k/n menunjukkan pasangan berubah/tersedia. "
    "TP: pasangan ada dan semua selisih nol; ND: pasangan tidak tersedia. Arsir k=1–19 adalah kehati-hatian. "
    "Format respons mencerminkan teks log yang tersimpan; tidak ditafsirkan sebagai struktur reasoning tersembunyi.",
    "",
    f"Holm tetap {CP_FAMILY_SIZE} hipotesis/model dari v2; pemecahan gambar tidak mengubah p, efek, CI, atau dukungan. "
    "CI karakteristik adalah bootstrap marginal; tidak ditampilkan untuk n<2. "
    "Segitiga memakai n<20, berbeda dari arsir heatmap yang memakai k<20. "
    "CI tidak mengatasi seleksi penjelasan yang tersedia. Penanda leksikal tidak mengukur kemampuan penalaran langsung.",
    "",
    "Gabungan ETHICS bersifat pendamping berbobot pasangan tersedia. Hasil tiap subset tetap tersedia. "
    "Perbandingan Man−Woman ditempatkan sebagai pelengkap; tidak ada seleksi berdasarkan signifikansi.",
    "", "## Menjalankan ulang", "",
    "Jalankan hanya tiga sel kode pada bagian **Presentasi Bab VI v3 — kelompok penanda** di notebook aktif. "
    "Setiap eksekusi menghasilkan run dan subfolder baru. Hindari menjalankan bagian rebuild legacy untuk sekadar merender gambar.",
    "",
    "Snapshot mandiri: presentation_notebook.ipynb. Eksekusi snapshot dari direktori repositori:",
    "",
    '    .\\.venv\\Scripts\\python.exe -m jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=3600 '
        f'--output executed_review_{CP_ID}.ipynb --output-dir "{CP_ACTIVE}" "{CP_RUN / "presentation_notebook.ipynb"}"',
    "", "## Caption",
]
for row in CP_INDEX.itertuples():
    cp_lines += ["",f"### {row.figure}","",row.caption]
cp_readme="\n".join(cp_lines)+"\n"
(CP_RUN/"README.md").write_text(cp_readme,encoding="utf-8")
(CP_DEST/"README.md").write_text(cp_readme,encoding="utf-8")
CP_INDEX.to_csv(CP_DEST/"figure_index.csv",index=False,encoding="utf-8-sig")
CP_POINTER=CP_ACTIVE/f"PRESENTASI_BAB6_V3_{CP_ID}.json"
assert not CP_POINTER.exists()
CP_POINTER.write_text(json.dumps(dict(run=str(CP_RUN),figures=str(CP_DEST),
    proof_pdf=str(CP_RUN/"pratinjau_A4_ukuran_asli.pdf"),manifest=str(CP_RUN/"manifest.json")),
    ensure_ascii=False,indent=2),encoding="utf-8")
print("SELESAI",CP_BENCHMARK,CP_RUN,flush=True)
print("GAMBAR",CP_DEST,flush=True)
print("PRATINJAU",CP_RUN/"pratinjau_A4_ukuran_asli.pdf",flush=True)
